In [14]:
import geopandas as gpd

# 1. Load your Burnie boundary polygon and the statewide SA1 layer
burnie_boundary = gpd.read_file(r"C:/Users/mpanta/Documents/AT3_KGG375/Burnie Shapefile/Burnie_Shapefile2.geojson")
sa1_gdf = gpd.read_file(r"C:/Users/mpanta/Documents/AT3_KGG375/Method of Travel to work/G62_TAS_GDA2020.gpkg", layer="G62_SA1_2021_TAS")

# 2. Ensure both datasets use the exact same Coordinate Reference System (CRS)
if burnie_boundary.crs != sa1_gdf.crs:
    burnie_boundary = burnie_boundary.to_crs(sa1_gdf.crs)

# 3. Clip the SA1 polygons strictly to the Burnie boundary
burnie_sa1 = gpd.clip(sa1_gdf, burnie_boundary)

# 4. Save the clipped file to your project folder
output_path = r"C:/Users/mpanta/Documents/AT3_KGG375/burnie_sa1_clipped.geojson"
burnie_sa1.to_file(output_path, driver="GeoJSON")

print(f"Successfully clipped and saved {len(burnie_sa1)} Burnie SA1 zones!")

Successfully clipped and saved 65 Burnie SA1 zones!


In [13]:
import geopandas as gpd

# 1. Load your GeoJSON file
input_path = r"C:\Users\mpanta\Documents\AT3_KGG375\burnie_sa1_clipped.geojson"
gdf = gpd.read_file(input_path)

# 2. Comprehensive list of terms/substrings to drop
drop_terms = [
    "Train",
    "Tram",
    "rail",
    "Truck",
    "Motorbike",
    "scootr",
    "Bicycle",
    "Bike",
    "Walked",
    "Tot_",
    "Total",
    "Tot-",
    "tot_one_method",
    "tot_two_methods",
    "tot_three_meth",
    "Othr",
    "Other",
    "Worked_home",
    "Did_not_go_to_work",
    "Method_travel_to_work_ns",
]

# Drop any column matching these terms (case-insensitive)
cols_to_drop = [
    col
    for col in gdf.columns
    if any(term.lower() in col.lower() for term in drop_terms)
]
gdf = gdf.drop(columns=cols_to_drop)

# 3. Define remaining valid multi-method travel category groups (using Persons _P suffix)
bus_multimodal_columns = [
    "Two_methods_Bus_Ferry_P",
    "Two_methods_Bus_Car_as_drvr_P",
    "Two_methods_Bus_Car_as_pass_P",
    "Two_methods_Bus_Other_P",
]

car_multimodal_columns = [
    "Two_methods_Bus_Car_as_drvr_P",
    "Two_methods_Bus_Car_as_pass_P",
]

# 4. Safely calculate totals for available columns
existing_bus_cols = [col for col in bus_multimodal_columns if col in gdf.columns]
existing_car_cols = [col for col in car_multimodal_columns if col in gdf.columns]

if existing_bus_cols:
    gdf["Total_Bus_Multimodal"] = gdf[existing_bus_cols].sum(axis=1)
if existing_car_cols:
    gdf["Total_Car_Multimodal"] = gdf[existing_car_cols].sum(axis=1)

all_multimodal_columns = list(set(existing_bus_cols + existing_car_cols))
if all_multimodal_columns:
    gdf["Total_Multimodal_Trips"] = gdf[all_multimodal_columns].sum(axis=1)

# 5. Save the cleaned GeoJSON file
output_path = r"C:\Users\mpanta\Documents\AT3_KGG375\burnie_sa1_transport_modes.geojson"
gdf.to_file(output_path, driver="GeoJSON")

print(
    f"Dropped {len(cols_to_drop)} unwanted columns. Saved updated file with {len(gdf)} features!"
)

Dropped 72 unwanted columns. Saved updated file with 20 features!
